# Data cleaning: raw instrument logs to merged CSVs

This notebook is the first step of the pipeline. It reads the 50 original instrument
files in `raw_data/` and rebuilds the three per-instrument tables that
`code/main.ipynb` starts from:

| Output | Instrument | Source files |
|---|---|---|
| `processed/merged_raft.csv` | RAFT heat-flux logger (heat flux + PT100) | `*_RAFT.txt` |
| `processed/merged_meteo.csv` | Weather station | `*_meteo.log` |
| `processed/merged_ir.csv` | Infrared sky thermometer | `*IR*.dat`, `*IR*.txt` |

Cleaning is limited to making the files readable and consistent. Values are kept as
recorded, apart from these changes:

- **Format fixes:** decimal commas become points, `\r`-only line endings are
  handled, and the weather station's control characters (STX/ETX) are removed.
- **One table per instrument:** each daily file is appended in file-name order.
  Every row records the file it came from.
- **Duplicate IR readings:** the IR `.dat` exports overlap (later files repeat
  earlier sessions). Identical readings are kept once, and `SOURCE_FILES` lists every
  file that contained them.
- **Nothing is dropped:** no row is removed for its value, no missing value is
  filled, and no unit is converted. Smoothing, binning and gap handling happen
  later, in `main.ipynb`.

The last section checks that each rebuilt table is byte-identical to the file in
`processed/`. This proves the committed data can be reproduced from the raw logs.

In [1]:
from pathlib import Path
import csv
import hashlib
import re
from datetime import datetime, timedelta

import pandas as pd

# Works whether the notebook is launched from code/ or from the project root.
ROOT = Path.cwd().resolve()
if not (ROOT / 'raw_data').exists():
    ROOT = ROOT.parent
RAW = ROOT / 'raw_data'
PROCESSED = ROOT / 'processed'

# False: rebuild in memory and compare only. True: overwrite the files in processed/.
WRITE_OUTPUTS = False

print(f'{len(list(RAW.iterdir()))} raw files in raw_data/')

50 raw files in raw_data/


## 1. RAFT heat-flux logger

The RAFT `.txt` files use bare carriage returns (`\r`) as line breaks and repeat a
three-line header every time acquisition restarts. Each data line holds the
heat-flux reading, the PT100 resistance, a date and a time. Restart marker lines,
which read `ND ND <date> <time>`, are kept as recorded, so `main.ipynb` sees exactly
what the logger wrote.

The `HEAT_FLUX_V` header does not guarantee volts on every row. `main.ipynb`
resolves the units, so this step leaves the values unchanged.

In [2]:
RAFT_LINE = re.compile(r'^\s*(\S+)\s+(\S+)\s+(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2}:\d{2})\s*$')

def read_raft(path):
    text = path.read_bytes().decode('latin-1')
    rows = []
    for line in re.split(r'\r\n|\r|\n', text):
        match = RAFT_LINE.match(line)
        if match:  # header lines contain words, so they never match
            heat_flux, pt100, date, time = match.groups()
            rows.append((heat_flux.replace(',', '.'), pt100.replace(',', '.'), date, time, path.name))
    return rows

raft_files = sorted(RAW.glob('*_RAFT.txt'))
raft = pd.DataFrame([row for f in raft_files for row in read_raft(f)],
                    columns=['HEAT_FLUX_V', 'PT100_VENT_OHM', 'DATE', 'TIME', 'SOURCE_FILE'])
print(f'{len(raft_files)} RAFT files -> {len(raft):,} rows '
      f'({raft.HEAT_FLUX_V.eq("ND").sum()} restart marker)')
raft.SOURCE_FILE.value_counts(sort=False).rename('rows').to_frame()

11 RAFT files -> 35,132 rows (1 restart marker)


,rows
SOURCE_FILE,
20260417_RAFT.txt,36
20260420_RAFT.txt,1405
20260421_RAFT.txt,1811
20260422_RAFT.txt,37
20260423_RAFT.txt,5755
20260427_RAFT.txt,1768
20260428_RAFT.txt,2603
20260430_RAFT.txt,5725
20260504_RAFT.txt,4338


## 2. Weather station

Each `.log` file has one header line followed by comma-separated readings. The
station wraps every message in STX (`\x02`) and ETX (`\x03`) control characters, and
adds a two-character checksum after the solar irradiance. The control characters are
removed. The checksum is kept as its own column.

In [3]:
METEO_COLUMNS = ['TIME', 'TEMP', 'DEWPOINT', 'RH', 'PRESS', 'SPEED', 'DIR', 'SOLARRAD', 'CHECKSUM']

def read_meteo(path):
    lines = re.split(r'\r\n|\r|\n', path.read_bytes().decode('latin-1'))
    rows = []
    for line in lines[1:]:  # the first line is the header
        if not line.strip():
            continue
        fields = [field.strip() for field in re.sub(r'[\x00-\x1f]', '', line).split(',')]
        if len(fields) != len(METEO_COLUMNS):
            raise ValueError(f'{path.name}: expected {len(METEO_COLUMNS)} fields, got {len(fields)}: {line!r}')
        rows.append(fields + [path.name])
    return rows

meteo_files = sorted(RAW.glob('*_meteo.log'))
meteo = pd.DataFrame([row for f in meteo_files for row in read_meteo(f)],
                     columns=METEO_COLUMNS + ['SOURCE_FILE'])
print(f'{len(meteo_files)} weather files -> {len(meteo):,} rows')
meteo.head()

11 weather files -> 35,327 rows

,TIME,TEMP,DEWPOINT,RH,PRESS,SPEED,DIR,SOLARRAD,CHECKSUM,SOURCE_FILE
0,Fri 17 April 2026 14:54:24,+022.7,+012.9,054,0986.7,002.36,178,0751,2A,20260417_meteo.log
1,Fri 17 April 2026 14:54:24,+023.0,+012.9,052,0986.7,002.80,220,0755,2D,20260417_meteo.log
2,Fri 17 April 2026 14:55:24,+022.3,+012.6,054,0986.6,002.32,244,0747,2F,20260417_meteo.log
3,Fri 17 April 2026 14:56:24,+022.7,+013.1,054,0986.7,001.61,104,0745,2C,20260417_meteo.log
4,Fri 17 April 2026 14:57:24,+023.1,+012.8,052,0986.8,000.09,144,0771,26,20260417_meteo.log


## 3. Infrared sky thermometer

The IR thermometer was exported in two formats:

- **Detailed `.dat` files** (20–28 April). These contain a metadata header (session
  date and time, unit, resolution, triggers) and 17 tab-separated columns per
  reading. Each reading's timestamp is the session start plus its elapsed time, plus
  the milliseconds from the absolute-time column.
- **Simple `.txt` files** (28 April onwards). Each line is a timestamp and a
  temperature.

The `.dat` exports overlap: a later file often repeats readings from an earlier
session. A reading is a duplicate only if its session and raw line are identical,
and then it is kept once. The same rule applies to simple readings with an identical
timestamp and temperature. All rows are then sorted by time.

This is a Python port of the earlier PowerShell script `archive/scripts/merge_ir.ps1`.
File names are sorted as PowerShell does, ignoring hyphens, so `SOURCE_FILES` comes
out identical.

In [4]:
DAT_COLUMNS = ['ELAPSED_TIME', 'TPROC', 'TINT', 'TBOX', 'TAVG', 'T_2C', 'T_1C', 'ATTENUATION', 'EPS',
               'VALUE', 'VCC', 'TAMB', 'TACT', 'COMPRESSION_IDX', 'TIME_ABSOLUTE', 'IMAGE_IDX', 'IMAGE_VALUE']
DAT_META = ['Date', 'Time', 'Unit', 'Resolution', 'Values', 'SnapshotTrigger', 'MeasurementTrigger']
IR_COLUMNS = ['RECORD_TYPE', 'TIMESTAMP', 'TEMPERATURE_C', *DAT_COLUMNS, 'SESSION_DATE', 'SESSION_TIME',
              'UNIT', 'RESOLUTION', 'VALUES_COUNT', 'SNAPSHOT_TRIGGER', 'MEASUREMENT_TRIGGER',
              'SOURCE_FILES', 'SOURCE_FILE_COUNT']

def ps_name_order(name):
    # PowerShell's culture-aware sort ignores hyphens: IR.dat sorts before IR-B.dat
    return name.lower().replace('-', '')

def decimal(value):
    return value.strip().replace(',', '.')

def ir_files(pattern):
    return sorted((f for f in RAW.glob(pattern) if 'ir' in f.name.lower()), key=lambda f: ps_name_order(f.name))

# Detailed .dat exports
detailed, detailed_lines = {}, 0
for path in ir_files('*.dat'):
    lines = path.read_bytes().decode('cp1252').splitlines()
    meta = {key: next((l[len(key) + 1:].strip() for l in lines if l.startswith(key + ':')), '') for key in DAT_META}
    session_start = datetime.strptime(f"{meta['Date']} {meta['Time']}", '%d/%m/%Y %H:%M:%S')
    for line in lines:
        if not re.match(r'^\d{3}:\d{2}:\d{2},\d{3}\t', line):
            continue
        detailed_lines += 1
        parts = line.split('\t')
        if len(parts) != 17:
            raise ValueError(f'{path.name}: expected 17 fields, got {len(parts)}')
        key = (meta['Date'], meta['Time'], line)
        if key in detailed:
            detailed[key]['sources'].add(path.name)
            continue
        h, m, s, ms = map(int, re.match(r'^(\d{3}):(\d{2}):(\d{2}),(\d{3})$', parts[0]).groups())
        absolute = re.match(r'^\d{2}:\d{2}:\d{2}:(\d{3})$', parts[14])
        timestamp = session_start + timedelta(hours=h, minutes=m, seconds=s,
                                              milliseconds=ms + (int(absolute.group(1)) if absolute else 0))
        values = [parts[0].replace(',', '.').strip()] + [
            parts[i].strip() if i == 14 else decimal(parts[i]) for i in range(1, 17)]
        detailed[key] = {'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3],
                         'values': values, 'meta': meta, 'sources': {path.name}}

# Simple .txt exports
SIMPLE_LINE = re.compile(r'^(\d{4}-\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2})\s+([+-]?\d+,\d+)\s*$')
simple, simple_lines = {}, 0
for path in ir_files('*.txt'):
    for line in path.read_bytes().decode('cp1252').splitlines():
        match = SIMPLE_LINE.match(line)
        if not match:
            continue
        simple_lines += 1
        key = (f'{match.group(1)} {match.group(2)}.000', decimal(match.group(3)))
        simple.setdefault(key, set()).add(path.name)

def sources(names):
    ordered = sorted(names, key=ps_name_order)
    return [';'.join(ordered), str(len(ordered))]

rows = []
for r in detailed.values():
    m = r['meta']
    rows.append(['DETAILED_DAT', r['timestamp'], '', *r['values'], m['Date'], m['Time'], m['Unit'],
                 m['Resolution'], m['Values'], m['SnapshotTrigger'], m['MeasurementTrigger'], *sources(r['sources'])])
for (timestamp, temperature), names in simple.items():
    rows.append(['SIMPLE_TXT', timestamp, temperature, *[''] * 17, '', '', 'degC', '', '', '', '', *sources(names)])

ir = (pd.DataFrame(rows, columns=IR_COLUMNS)
        .sort_values(['TIMESTAMP', 'RECORD_TYPE'], kind='stable').reset_index(drop=True))
print(f'IR table: {len(ir):,} rows, {ir.TIMESTAMP.min()} -> {ir.TIMESTAMP.max()}')
pd.DataFrame({'input lines': [detailed_lines, simple_lines],
              'unique readings': [len(detailed), len(simple)],
              'duplicates removed': [detailed_lines - len(detailed), simple_lines - len(simple)]},
             index=['detailed .dat', 'simple .txt'])

IR table: 33,890 rows, 2026-04-20 09:36:08.788 -> 2026-05-15 12:03:23.000


,input lines,unique readings,duplicates removed
detailed .dat,20422,9573,10849
simple .txt,24317,24317,0


## 4. Verify against `processed/` and optionally save

Each table is written the same way as the committed files: every field quoted,
UTF-8 with BOM, and Windows line endings. The SHA-256 hash is then compared with the
file in `processed/`. A match means the committed file is exactly what the raw logs
produce. Set `WRITE_OUTPUTS = True` in the first code cell to regenerate the files.

In [5]:
def to_csv_bytes(frame):
    return frame.to_csv(index=False, quoting=csv.QUOTE_ALL, lineterminator='\r\n').encode('utf-8-sig')

checks = []
for name, frame in [('merged_raft.csv', raft), ('merged_meteo.csv', meteo), ('merged_ir.csv', ir)]:
    rebuilt = to_csv_bytes(frame)
    target = PROCESSED / name
    committed = target.read_bytes() if target.exists() else b''
    identical = hashlib.sha256(rebuilt).digest() == hashlib.sha256(committed).digest()
    if WRITE_OUTPUTS:
        target.write_bytes(rebuilt)
    checks.append({'file': name, 'rows': len(frame), 'columns': frame.shape[1],
                   'sha256': hashlib.sha256(rebuilt).hexdigest()[:16] + '...',
                   'identical to processed/': identical})

checks = pd.DataFrame(checks).set_index('file')
assert checks['identical to processed/'].all() or WRITE_OUTPUTS, 'Rebuilt data differ from processed/'
checks

,rows,columns,sha256,identical to processed/
file,,,,
merged_raft.csv,35132,5,6fb1a55e183d4c03...,True
merged_meteo.csv,35327,10,fc8057c64cc3b139...,True
merged_ir.csv,33890,29,81a249802d48274a...,True


## 5. Where `merged_dataset_104957.csv` comes from

`main.ipynb` also reads `processed/merged_dataset_104957.csv`, but only for comparing
acquisitions and for the RAFT unit metadata (`Q_raw_unit`). This table was not
produced by this notebook. It is byte-identical to `Data/outputs/merged_dataset.csv`
from Dipu bhai's original pipeline on the `old_work` branch (`reader.py`,
`merger.py`, `main.py`).

That pipeline put RAFT on the master timeline. It then attached the nearest weather
and IR readings within ±90 s (`pd.merge_asof`, `direction='nearest'`), converted
heat-flux voltage to W/m² and PT100 resistance to °C, and covered raw files up to
3 July 2026. Files after 15 May are not in `raw_data/`, so this table cannot be
rebuilt here. Its hash is recorded in `data_manifest.json`, and `main.ipynb` checks
it before use.